In [0]:
folder_path = '/Volumes/ai_capabilities/document_intelligence/invoices/load_invoice_1/test_invoices/test_invoices/'

In [0]:
from pyspark.sql import functions as F

binary_df = spark.read.format("binaryFile").load(f"{folder_path}invoice_B_moretti.pdf")

df = binary_df.select(F.expr("ai_parse_document(content, MAP('version', '2.0')) AS parsed"))
display(df)

In [0]:
from pyspark.sql import functions as F

binary_df = spark.read.format("binaryFile").load(
    f"{folder_path}*"
)

parsed_df = (
    binary_df
    .filter(F.col("path").endswith(".pdf"))
    .select("path", F.expr("ai_parse_document(content, map('version','2.0')) AS parsed"))
)

parsed_df.write.mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.bronze_invoices_parsed")

bronze_df = spark.table("ai_capabilities.document_intelligence.bronze_invoices_parsed")

errors_df = bronze_df.select("path", F.expr("parsed:error_status AS errors"))
display(errors_df)


confidence_df = (
    bronze_df.alias("b")
    .select("b.path", F.explode(F.expr("try_cast(parsed:document:elements AS ARRAY<VARIANT>)")).alias("e"))
    .select("path", F.expr("e:confidence::double AS confidence"))
    .groupBy("path")
    .agg(
        F.count("*").alias("elements"),
        F.min("confidence").alias("min_conf"),
        F.round(F.avg("confidence"), 3).alias("avg_conf"),
    )
)
display(confidence_df)

In [0]:
from pyspark.sql import functions as F

binary_df = spark.read.format("binaryFile").load(
    f"{folder_path}*"
)

parsed_df = (
    binary_df
    .filter(F.col("path").endswith(".pdf"))
    .select("path", F.expr("ai_parse_document(content, map('version','2.0')) AS parsed"))
)
extract_df = parsed_df.select("path", F.expr("""  ai_extract(
    parsed,
    '{
      "invoice_number": {"type":"string","description":"Invoice number of the seller invoice"},
      "invoice_date":   {"type":"string","description":"Invoice date, YYYY-MM-DD"},
      "vendor_vat_id":  {"type":"string","description":"VAT ID of the seller (issuer)"},
      "customer_vat_id":{"type":"string","description":"VAT ID of the customer (recipient)"},
      "total_amount":   {"type":"number","description":"Final total amount payable in EUR"},
      "line_items": {"type":"array","items":{"type":"object","properties":{
         "pos":{"type":"string"},"art_no":{"type":"string"},"description":{"type":"string"},
         "qty":{"type":"number"},"unit":{"type":"string"},
         "unit_price":{"type":"number"},"amount":{"type":"number"}}}}
    }',
    map('version','2.1','mode','precision','enableConfidenceScores','true',
        'instructions','Vendor invoices. Ignore stamps and handwritten notes. Do not output the Carried forward or Carried over rows as line items.')
  ) AS extracted
"""))
display(extract_df)
extract_df.write.mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.silver_invoices_extracted")



In [0]:
from pyspark.sql import functions as F


# 6) Build gold from the SAVED silver table (no AI calls here)
s = (spark.table("ai_capabilities.document_intelligence.silver_invoices_extracted")
       .withColumn("file_name", F.element_at(F.split("path", "/"), -1)))
# ...then your existing header / lines code and the two saveAsTable writes, unchanged
# s = extract_df.withColumn("file_name", F.element_at(F.split("path", "/"), -1))

# 1) Header: one row per invoice
header = s.select(
    "file_name",
    F.expr("extracted:response:invoice_number:value::string").alias("invoice_number"),
    F.to_date(F.expr("extracted:response:invoice_date:value::string")).alias("invoice_date"),
    F.expr("extracted:response:vendor_vat_id:value::string").alias("vendor_vat_id"),
    F.expr("extracted:response:customer_vat_id:value::string").alias("customer_vat_id"),
    F.expr("extracted:response:total_amount:value::decimal(18,2)").alias("total_amount"),
    F.expr("extracted:error_message::string").alias("error_message"),
)

# 2) Lines: one row per line item
lines = (
    s.select("file_name",
             F.posexplode(F.expr("try_cast(extracted:response:line_items AS ARRAY<VARIANT>)")).alias("idx", "li"))
     .select(
        "file_name",
        (F.col("idx") + 1).alias("line_no"),
        F.expr("li:pos:value::string").alias("pos"),
        F.expr("li:art_no:value::string").alias("art_no"),
        F.regexp_replace(
            F.regexp_replace(F.expr("li:description:value::string"), "\u00b5", "\u03bc"),  # unify µ / μ
            r"\s*\n\s*", " ").alias("description"),
        F.expr("li:qty:value::decimal(18,2)").alias("qty"),
        F.expr("li:unit:value::string").alias("unit"),
        F.expr("li:unit_price:value::decimal(18,4)").alias("unit_price"),
        F.expr("li:amount:value::decimal(18,2)").alias("amount"),
    )
)

header.write.mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.gold_invoice_header")
lines.write.mode("overwrite").saveAsTable("ai_capabilities.document_intelligence.gold_invoice_lines")

In [0]:
%sql
SELECT h.file_name, h.total_amount, count(*) AS n_lines,
       sum(l.amount) AS lines_sum, h.total_amount - sum(l.amount) AS diff
FROM ai_capabilities.document_intelligence.gold_invoice_header h
JOIN ai_capabilities.document_intelligence.gold_invoice_lines l ON h.file_name = l.file_name
GROUP BY h.file_name, h.total_amount;

In [0]:
%sql
CREATE OR REPLACE TABLE ai_capabilities.document_intelligence.gold_invoice_validation AS
SELECT h.file_name, h.invoice_number, h.total_amount,
       count(l.line_no) AS n_lines, sum(l.amount) AS lines_sum,
       h.total_amount - sum(l.amount) AS diff,
       CASE WHEN h.error_message IS NOT NULL THEN 'EXTRACT_ERROR'
            WHEN count(l.line_no) = 0 THEN 'NO_LINES'
            WHEN h.total_amount IS NULL OR h.invoice_number IS NULL OR h.invoice_date IS NULL THEN 'MISSING_FIELD'
            WHEN abs(h.total_amount - sum(l.amount)) > 0.01 THEN 'TOTAL_MISMATCH'
            ELSE 'OK' END AS status
FROM ai_capabilities.document_intelligence.gold_invoice_header h
LEFT JOIN ai_capabilities.document_intelligence.gold_invoice_lines l USING (file_name)
GROUP BY h.file_name, h.invoice_number, h.total_amount, h.invoice_date, h.error_message;
SELECT * FROM ai_capabilities.document_intelligence.gold_invoice_validation;

In [0]:
%sql
SELECT 'header' AS t, count(*) AS n FROM ai_capabilities.document_intelligence.gold_invoice_header
UNION ALL SELECT 'lines', count(*) FROM ai_capabilities.document_intelligence.gold_invoice_lines
UNION ALL SELECT 'validation', count(*) FROM ai_capabilities.document_intelligence.gold_invoice_validation;

# Pipeline overview

Bronze → silver → gold IDP pipeline over the test invoice PDFs, with validation at the end.

* **Cell 1 — Source path setup.** Defines `folder_path` pointing at the Unity Catalog volume holding the test invoices. Every file read in Cells 2–5 uses this variable.
* **Cell 2 — Single-file parse demo (SQL).** Parses one PDF (`invoice_B_moretti.pdf`) with `ai_parse_document` via `spark.sql` + `READ_FILES` and displays the raw parsed variant — an exploratory look at the parse output that Cell 3 reproduces in PySpark.
* **Cell 3 — Single-file parse demo (PySpark).** Same single-PDF parse via the DataFrame API, with a commented-out sketch of exploding `parsed:document:elements` into typed columns. It establishes the parse pattern that Cell 4 scales to the whole folder.
* **Cell 4 — Bronze layer.** Parses every `.pdf` in the folder and overwrites the bronze table `bronze_invoices_parsed`, then displays per-file parse errors (`parsed:error_status`) and per-file element-confidence stats. This bronze table is the shared input for the classification/extraction pipelines across the related notebooks.
* **Cell 5 — Silver layer.** Re-parses the PDFs and runs `ai_extract` with the invoice schema (precision mode, confidence scores, vendor-invoice instructions), overwriting `silver_invoices_extracted` with `path` + `extracted` per file. Its output table is exactly what Cell 6 consumes.
* **Cell 6 — Gold layer (no AI calls).** Reads the silver table and splits it into `gold_invoice_header` (one row per invoice) and `gold_invoice_lines` (one row per line item, exploded with line numbers, µ/μ unified). These two tables feed the validation in Cells 7–9.
* **Cell 7 — Reconciliation check.** SQL join of gold header + lines comparing `total_amount` against the sum of line amounts (`diff`) per file. An interactive check; Cell 8 formalizes it into a persisted status.
* **Cell 8 — Validation table.** `CREATE OR REPLACE TABLE gold_invoice_validation` — left-joins header and lines, assigns each invoice a status (`EXTRACT_ERROR`, `NO_LINES`, `MISSING_FIELD`, `TOTAL_MISMATCH`, `OK`), then displays the result. This is the pipeline's quality gate.
* **Cell 9 — Sanity counts.** Row counts across `gold_invoice_header`, `gold_invoice_lines`, and `gold_invoice_validation` via `UNION ALL` — a quick completeness check that the three gold tables agree.

**Overall flow:** Cell 1 (path) → Cells 2–3 (single-file demos) → Cell 4 (bronze: parse all) → Cell 5 (silver: extract) → Cell 6 (gold: header + lines) → Cells 7–8 (validate) → Cell 9 (counts).